# Policy Iteration on the Logistics Grid MDP

This notebook demonstrates:
1. Setting up the `LogisticsGridMDP` environment
2. Training a `DPAgent` via policy iteration
3. Visualizing the learned policy map and value heatmap
4. Evaluating the agent over multiple episodes

In [ ]:
import sys
import os

# Add project root to path so src/ imports work from notebooks/
sys.path.insert(0, os.path.abspath(".."))

## 1. Environment Setup

In [ ]:
from src.envs.logistics_grid_mdp import LogisticsGridMDP, LogisticsGridSpec

# Default spec: 6x6 grid, depot at (0,0), customer at (5,5), slip_prob=0.10
spec = LogisticsGridSpec()
mdp = LogisticsGridMDP(spec)

print(f"States : {mdp.nS}  (including blocked)")
print(f"Actions: {mdp.nA}  -> {mdp.ACTIONS}")
print(f"Depot  : state {mdp.start_state()}")
print(f"Goal   : state {mdp.goal_state()}")
print(f"Gamma  : {mdp.gamma}")
print(f"Slip   : {spec.slip_prob}")

## 2. Train the DPAgent

In [ ]:
from src.agents.dp_agent import DPAgent

agent = DPAgent(
    mdp=mdp,
    gamma=0.95,
    theta=1e-8,
    max_outer_iter=1000
)

result = agent.train()

## 3. Inspect Learned Values

In [ ]:
import numpy as np

print(f"Value at depot  (state {mdp.start_state()}): {agent.V[mdp.start_state()]:.4f}")
print(f"Value at goal   (state {mdp.goal_state()}):  {agent.V[mdp.goal_state()]:.4f}")
print(f"\nQ-values at depot (state {mdp.start_state()}):")
for a, name in enumerate(mdp.ACTIONS):
    print(f"  {name:6s}: {agent.Q[mdp.start_state(), a]:.4f}")

## 4. Policy Map (ASCII)

In [ ]:
from src.utils.viz import policy_to_grid, print_policy_grid

grid = policy_to_grid(
    policy=agent.policy,
    rows=mdp.rows,
    cols=mdp.cols,
    blocked_states=mdp._blocked_states,
    terminal_states=mdp._terminal_states
)

print("Policy Map  (D=Depot  G=Goal  █=Blocked)\n")
print_policy_grid(grid, depot_rc=spec.depot, goal_rc=spec.customer)

## 5. Value Function Heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

grid_V = agent.V.reshape(mdp.rows, mdp.cols).copy()
for s in mdp._blocked_states:
    r, c = divmod(s, mdp.cols)
    grid_V[r, c] = np.nan

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(grid_V, cmap="viridis")
fig.colorbar(im, ax=ax, label="V(s)")
ax.set_title("Value Function Heatmap")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.tight_layout()
plt.show()

## 6. Evaluate the Agent

In [ ]:
returns = agent.evaluate(episodes=20, max_steps=500)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(returns, marker="o")
ax.axhline(np.mean(returns), color="red", linestyle="--", label=f"Mean = {np.mean(returns):.3f}")
ax.set_xlabel("Episode")
ax.set_ylabel("Discounted Return")
ax.set_title("DPAgent Evaluation Returns")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Save Outputs

In [ ]:
agent.save_policy("../outputs/policies/best_policy.json")
agent.save_values("../outputs/values/V.npy", "../outputs/qvalues/Q.npy")
print("Saved policy, V, and Q to outputs/")